In [ ]:
# def ruoff_model(y, t, J0, k1, k2, k3, k4, k5, k6, k, kappa, q, K1, psi):
#     # Pooled variable totals
#     N = 1.0
#     A = 4.0

#     # Unpack y
#     S1, S2, S3, S4, N2, A3, S5 = y

#     # Reaction rates
#     v1 = k1 * S1 * A3 / (1 + (A3 / K1)**q)
#     v2 = k2 * S2 * (N - N2)
#     v3 = k3 * S3 * (A - A3)
#     v4 = k4 * S4 * N2
#     v5 = k5 * A3
#     v6 = k6 * S2 * N2
#     v7 = k * S5
#     J = kappa * (S4 - S5)

#     # ODEs
#     dS1dt = J0 - v1
#     dS2dt = 2*v1 - v2 - v6
#     dS3dt = v2 - v3
#     dS4dt = v3 - v4 - J
#     dN2dt = v2 - v4 - v6
#     dA3dt = -2*v1 + 2*v3 - v5
#     dS5dt = psi * J - v7

#     return [dS1dt, dS2dt, dS3dt, dS4dt, dN2dt, dA3dt, dS5dt]


# # Constants
# Tref = 286.5  # K
# R = 0.0083144  # kJ/K/mol
# Temperature = 298.15

# # Prefactors at Tref (units mM/min except K1 in mM)
# x_temp = np.array([
#     2.5,
#     100.,
#     6.0,
#     16.0,
#     100.0,
#     1.28,
#     12.0,
#     1.8,
#     13.0,
#     0.52
# ])

# Ebarriers = np.array([
#     16.2,  # EJ0
#     13.8,  # Ek1
#     60.7,  # Ek2
#     44.9,  # Ek3
#     58.7,  # Ek4
#     41.2,  # Ek5
#     31.4,  # Ek6
#     24.3,  # Ek
#     15.9,  # Ekappa
#     47.0   # EK1
# ])

# # Temperature-adjusted rate constants
# prefactors = x_temp * np.exp(Ebarriers / (R * Tref))
# x_temp_new = prefactors * np.exp(-Ebarriers / (R * Temperature))

# # Assign rate constants
# J0, k1, k2, k3, k4, k5, k6, k, kappa, K1 = x_temp_new
# q = 4.0
# psi = 0.1

# # Parameters passed to the ODE function
# params = (J0, k1, k2, k3, k4, k5, k6, k, kappa, q, K1, psi)

# t_span = np.linspace(0, 5, 1000)
# yzero = [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1]

# data = generate_ode_data(system=ruoff_model)          # Data from the system

# X_sol = data.solve_system(yzero, t_span, args=params)

# X, X_dot = data.data_matrices(X_sol, 3, args=params)

# #============================================================================
# def psi_ruoff(X, K1):
#     """
#     Returns transformed data matrices using basis functions relevant to the Ruoff model,
#     including nonlinear and rational terms.
#     X is a 7 x N array where each row corresponds to:
#     [S1, S2, S3, S4, N2, A3, S5]
#     """

#     # Constants from Ruoff model
#     #K1 = 1.12
#     q = 4.0

#     # Unpack species
#     S1 = X[0, :]
#     S2 = X[1, :]
#     S3 = X[2, :]
#     S4 = X[3, :]
#     N2 = X[4, :]
#     A3 = X[5, :]
#     S5 = X[6, :]

#     # Rational term from v1: S1 * A3 / (1 + (A3 / K1)^q)
#     v1_term = S1 * A3 / (1 + (A3 / K1)**q)

#     # Construct basis functions
#     features = np.vstack([
#         np.ones_like(S1),         # Constant term
#         S1, S2, S3, S4, N2, A3, S5,  # Linear terms

#         S1*A3,                    # v1
#         S2*(1 - N2),              # v2
#         S3*(4.0 - A3),            # v3
#         S4*N2,                    # v4
#         A3,                       # v5
#         S2*N2,                    # v6
#         S5,                       # v7
#         S4 - S5,                  # J = kappa*(S4 - S5)

#         v1_term,                 # Rational term from v1

#         S1**2, S2**2, A3**2, S5**2,   # Quadratic terms
#         S1*S2, S1*S3, S2*S3, S3*S4,   # Interaction terms
#         A3**3, S5**3,                 # Cubic terms
#         S1*A3**2, S1**2*A3,           # Mixed nonlinear terms
#         S1**3, S2**3, S3**3           # Higher-order terms
#     ])

#     return features


# psi_x = psi_ruoff(X, K1=1.12416028)
# xi_hr = sindy_ode(X, X_dot, psi_x)
# X_sindy = xi_hr @ psi_x


# K1_vals = np.linspace(100, 500, 50)
# costs = []
# for i in K1_vals:
#     psi_x_exp = psi_ruoff(X, K1=i)
#     xi_hr_exp = sindy_ode(X, X_dot, psi_x_exp)
#     X_sindy_exp = xi_hr_exp @ psi_x_exp
#     costs.append(cost_sindy(xi_hr_exp, X_dot, psi_x_exp))

In [ ]:
# ## lorenz63

# def lorenz63(y, t):
#     x, y, z = y

#     sigma, rho, beta = 10, 28, 8/3
#     alpha = 5

#     # Equations for the Lorenz system
#     xdot = sigma * (y - x) + np.tanh(alpha * x)
#     ydot = x * (rho - z) - y
#     zdot = x * y - beta * z
    
#     sys = np.array([xdot, ydot, zdot])
#     return sys


# def psi_lorenz63(X, alpha):
#     """Returns transformed data matrices using parametric set of basis functions, including Lorenz63 terms and extra terms."""
#     return np.vstack(
#         (
#             np.ones(X.shape[1]),  
#             X[0, :],              
#             X[1, :],              
#             X[2, :],              
#             X[0, :] * X[1, :],    
#             X[1, :] ** 3,         # y^3
#             X[0, :] * X[2, :] ** 2,
#             np.tanh(alpha*X[0, :]), 
#             X[0, :] ** 2,         
#             X[0, :] ** 3,         
#             X[1, :] ** 2,         
#             X[0, :] * X[2, :],    
#             X[0, :] ** 2 * X[1, :],
#             X[0, :] * X[1, :] * X[2, :] 
#         )
#     )
# psi_x = psi_lorenz63(X, alpha=1)
# xi_lrnz = sindy_ode(X, X_dot, psi_x)
# X_sindy = xi_lrnz @ psi_x     # reconstruction


# def Derivatives_lrnz(y, t, alpha=1):
#     x1, x2, x3 = y


#     basis = np.array([
#         np.ones_like(x1),               # 1
#         x1,                             # x1
#         x2,                             # x2
#         x3,                             # x3
#         x1 * x2,                        # x1 * x2
#         x2**3,                          # x2^3
#         x1 * x3**2,                     # x1 * x3^2
#         np.tanh(alpha * x1),             # sin(alpha * x1)
#         x1**2,                          # x1^2
#         x1**3,                          # x1^3
#         x2**2,                          # x2^2
#         x1 * x3,                        # x1 * x3
#         x1**2 * x2,                     # x1^2 * x2
#         x1 * x2 * x3                    # x1 * x2 * x3
#     ])


#     D = np.dot(xi_lrnz, basis)

#     return D


# X_sindy = odeint(Derivatives_lrnz, yzero, t_span).T # make shape (d, m) 

# print('Cost SINDy: ', cost_sindy(xi_lrnz, X_dot, psi_x))

In [ ]:
# ## Duffing Oscillator



# def system1(y, t):
#     x1, x2 = y
    
#     # parameters
#     m0, m1, e, omega, gamma = -0.0845, 0.66, 0.25, 10, 0.14

#     xdot = x2
#     ydot = (
#         -m1 * x1
#         - x1**3
#         + gamma * np.cos(omega * t)
#     )
#     # ydot = (
#     #     -m1 * x1
#     #     - (1 / 2 * (m0 - m1)) * (abs(x1 + 1) - abs(x1 - 1))
#     #     - e * x2
#     #     + gamma * np.cos(omega * t)
#     # )
    
#     sys = np.array([xdot, ydot])
#     return sys


# def psi_system1(X, t, omega):
#     """Returns transformed data matrices using parametric set of basis functions, including system-specific terms."""
#     return np.vstack(
#         (
#             np.ones(X.shape[1]),  
#             X[0, :],               # x1
#             X[1, :],               # x2
#             X[0, :] * X[1, :],     # x1 * x2
#             X[1, :] ** 2,          # x2^2
#             X[0, :] ** 2,          
#             X[0, :] ** 3,          
#             #np.abs(X[0, :] + 1) - np.abs(X[0, :] - 1),  
#             np.sin(X[0, :]), 
#             np.cos(omega * t), 
#             X[1, :] * X[0, :] ** 2,  
#             X[0, :] ** 2 * X[1, :],   
#             X[0, :] ** 2 * X[1, :]**2
#         )
#     )

# # Create time domain
# m = 1000                           # of data points
# #dt = 1e-3
# t_span = np.linspace(0, 10, m)     # time-span of solution 


# # Initial condition
# x0 = 0.1                             # Initial concentration x
# y0 = 0.1                               # Initial concentration y
# #z0 = 0.1                               # Initial concentration z
# yzero = [x0, y0]

# data = generate_ode_data(system=system1)          # Data from the system

# X_sol = data.solve_system(yzero, t_span)

# X, X_dot = data.data_matrices(X_sol, t_span)
# psi_x = psi_system1(X, t_span, omega=10)
# xi_lrnz = sindy_ode(X, X_dot, psi_x)
# #X_sindy = xi_lrnz @ psi_x     # reconstruction


# def Derivatives_sys1(y, t, omega=10):
#     x1, x2 = y

#     basis = np.array([
#         np.ones_like(x1),
#         x1,
#         x2,
#         x1 * x2,
#         x2**2,
#         x1**2,
#         x1**3,
#         #np.abs(x1 + 1) - np.abs(x1 - 1),
#         np.sin(x1),
#         np.cos(omega * t),
#         x2 * x1**2,
#         x1**2 * x2,
#         x1**2 * x2**2
#     ])

#     D = np.dot(xi_lrnz, basis)

#     return D


# X_sindy = odeint(Derivatives_sys1, yzero, t_span).T # make shape (d, m) 

# print('Cost SINDy: ', cost_sindy(xi_lrnz, X_dot, psi_x))

In [ ]:
# ## bio_gene_net



# def bio_gene_net(y, t):

#     x1, x2 = y
    
#     # Parameters
#     a1, a2, a3 = 1.5, 0.7, 12.4
#     b1, b2 = 0.82, 15.0

#     xdot = a1 + (a2 * x1**2) / (a3 + x1**2) - x1 / (1 + x1 + x2)
#     ydot = b1 / (1 + b2 * x1**5) - x2 / (1 + x1 + x2)

#     sys = np.array([xdot, ydot])
#     return sys

# # Create time domain
# m = 10000                           # of data points
# #dt = 1e-3
# t_span = np.linspace(0, 1000, m)     # time-span of solution 


# # Initial condition
# x0 = 0.1                             # Initial concentration x
# y0 = 0.                               # Initial concentration y
# #z0 = 0.                               # Initial concentration z
# yzero = [x0, y0]

# data = generate_ode_data(system=bio_gene_net)          # Data from the system

# X_sol = data.solve_system(yzero, t_span)

# X, X_dot = data.data_matrices(X_sol, 3)


# def psi_bio_gene_net(X, a3):
#     return np.vstack(
#         (
#             np.ones(X.shape[1]),
#             X[0, :],
#             X[1, :],
#             X[0, :] * X[1, :],
#             X[0, :] ** 2,
#             X[1, :] ** 2,
#             X[0, :] ** 3,
#             X[1, :] ** 3,
#             X[0, :] * X[1, :] ** 2,
#             X[0, :] ** 2 * X[1, :],
#             X[0, :] * X[1, :] ** 3,
#             X[0, :] * X[1, :],
#             (X[0, :] ** 2) / (X[0, :] ** 2 + a3),  # a2 * x1^2 / (a3 + x1^2)
#             X[0, :] / (1 + X[0, :] + X[1, :]),     # x1 / (1 + x1 + x2)
#             0.82 / (1 + 15.0 * X[0, :] ** 5),       # b1 / (1 + b2 * x1^5)
#             X[1, :] / (1 + X[0, :] + X[1, :])      # x2 / (1 + x1 + x2)
#         )
#     )

# a3 = -40.1

# psi_x = psi_bio_gene_net(X, a3)
# print(psi_x.shape) 
# xi_biog = sindy_ode(X, X_dot, psi_x)


# def Derivatives_bio_gene_net(y, t, a3=a3):
#     x1, x2 = y
#     basis = np.array([
#         np.ones_like(x1),
#         x1,
#         x2,
#         x1 * x2,
#         x1**2,
#         x2**2,
#         x1**3,
#         x2**3,
#         x1 * x2**2,
#         x1**2 * x2,
#         x1 * x2**3,
#         x1 * x2,
#         (x1**2) / (x1**2 + a3),               # a2 * x1^2 / (a3 + x1^2)
#         x1 / (1 + x1 + x2),                   # x1 / (1 + x1 + x2)
#         0.82 / (1 + 15.0 * x1**5),            # b1 / (1 + b2 * x1^5)
#         x2 / (1 + x1 + x2)                    # x2 / (1 + x1 + x2)
#     ])
#     D = np.dot(xi_biog, basis)
#     return D

# # Solve the system using odeint (simulating the behavior of the system)
# X_sindy = odeint(Derivatives_bio_gene_net, yzero, t_span).T

# print('Cost SINDy: ', cost_sindy(xi_biog, X_dot, psi_x))


# alpha_vals = np.linspace(-1, 7, 100)

# costs = []
# for i in alpha_vals:
#     psi_x_exp = psi_bio_gene_net(X, a3=i)
#     xi_lrnz_exp = sindy_ode(X, X_dot, psi_x_exp)
#     X_sindy_exp = xi_lrnz_exp @ psi_x_exp
#     costs.append(cost_sindy(xi_lrnz_exp, X_dot, psi_x_exp))


